# [Deployment](https://rhindottire.github.io/Data-Mining/dataProcess.html#deployment)

This stage turns the winning route from the Modeling page — skip-gram Word2Vec pooled by mean and classified by Gaussian naive bayes — into a standalone service, verifies the serving pipeline reproduces the Data Preparation transform, publishes the classifier as a public Gradio app on a Hugging Face Space, and closes the CRISP-DM loop. Only stored artifacts are used; nothing is crawled or re-cleaned here.

## 1. Plan Deployment

The winning arm reads raw article text at request time, so the serving pipeline must replay the exact cleaning sequence of the Data Preparation stage. That sequence lives in `deploy/serve.py`: ascii normalization (strip accents, map punctuation, drop non-latin symbols), artifact and number rules, sentence split, then token-by-token lowercase with slang normalization guarded by the protected-name list. Words absent from the skip-gram vocabulary are skipped, the remaining vectors are averaged, and the Gaussian naive bayes estimator returns the predicted class with the two class scores.

The app is a Gradio interface hosted on a free Hugging Face ZeroGPU Space with Python 3.12. It accepts a news URL — fetched with `trafilatura`, an honest user agent, at least one second between requests, stopping and recording any 403/429 refusal — or pasted text.

In [1]:
import json
import sys
from pathlib import Path

import pandas as pd

DEPLOY = Path('..') / '..' / 'deploy'
sys.path.insert(0, str(DEPLOY))

from serve import classify, describe

files = sorted(p.name for p in (DEPLOY / 'models').iterdir())
labels = json.loads((DEPLOY / 'models' / 'labels.json').read_text())
protected = len(json.loads((DEPLOY / 'models' / 'protected.json').read_text()))
sample = describe('Imbal hasil obligasi negara menguat di tengah rilis data neraca dagang')
result = pd.Series(
    {
        'packaged files': ', '.join(files),
        'class order': labels,
        'protected names': protected,
        'sample sentences': sample['sentences'],
        'sample tokens': sample['tokens'],
        'sample words in vocabulary': sample['words in vocabulary'],
    },
    name='package',
)
result

packaged files                gnb.joblib, labels.json, protected.json, word2...
class order                                                    [finance, sport]
protected names                                                            1274
sample sentences                                                              1
sample tokens                                                                11
sample words in vocabulary                                                    9
Name: package, dtype: object

The release package holds four artifacts: the skip-gram model, the naive bayes estimator, the class order, and the protected-name list that keeps proper nouns untouched during slang normalization. `describe()` replays the serving transform on one sentence and reports how many tokens land in the word2vec vocabulary.

In [2]:
from sklearn.model_selection import train_test_split

DATA = Path('..') / '..' / 'data' / 'Web-Mining'
raw = pd.read_csv(DATA / 'crawling_detik.csv')
_train_idx, test_idx = train_test_split(
    list(range(len(raw))),
    test_size=0.2,
    random_state=42,
    stratify=raw['label'],
)
hits = 0
for i in test_idx:
    label, scores = classify(raw['isi_berita'].iloc[i])
    hits += int(label == raw['label'].iloc[i])
pd.Series(
    {
        'test documents': len(test_idx),
        'matched predictions': hits,
        'accuracy': round(hits / len(test_idx), 3),
        'expected from Modeling': 0.975,
    },
    name='served parity',
)

test documents            40.000
matched predictions       39.000
accuracy                   0.975
expected from Modeling     0.975
Name: served parity, dtype: float64

The served pipeline is measured on the same stratified 160/40 split with the same seed the Modeling page fixed before any fit. The reproduced accuracy on the 40 held-out documents is the value the Modeling page reported, which means packing the estimator and replaying the transform does not drift the classifier.

In [3]:
import json
import urllib.request

with urllib.request.urlopen(
    'https://huggingface.co/api/spaces/Rhindottire/News-Classifier',
    timeout=30,
) as response:
    info = json.load(response)
runtime = info.get('runtime', {})
pd.Series(
    {
        'space': 'Rhindottire/News-Classifier',
        'app url': 'https://' + info['subdomain'] + '.hf.space',
        'stage': runtime.get('stage'),
        'hardware': runtime.get('hardware', {}).get('requested'),
    },
    name='released',
)

space                        Rhindottire/News-Classifier
app url     https://rhindottire-news-classifier.hf.space
stage                                            RUNNING
hardware                                       zero-a10g
Name: released, dtype: str

The public record of the Space reports the running stage and the ZeroGPU hardware. The deployed app is live; its interface mirrors the plan above and reads exactly the artifacts this notebook loads.

## 2. Produce Final Report

The released classifier is the same estimator the Modeling page validated. Any visitor opens the public URL, pastes a news link or raw text, and receives the predicted class, the two class scores, and a short sentence/token/vocabulary summary. The deployment reproduces the training transform (served parity above), performs no re-crawling, and applies the same polite-request rules for URL inputs as the collection stage. Final quality figures match Modeling: held-out accuracy 0.975, with sport F1 0.974 and finance F1 0.976 on the 40 held-out documents.

## 3. Review Project

What worked: keeping the service dependency-light, reusing one transform for training and serving, and verifying text equivalence with a parity script before going live. Constraints: Hugging Face now keeps ordinary Gradio Spaces behind a paid plan, so the free route is the ZeroGPU hardware, which requires one `@spaces.GPU`-decorated function even for a CPU-only classifier; the corpus covers 200 detik.com articles (sport/finance), so the app is sharpest on that kind of news. Next steps: retrain on a larger labeled set, extend to a third class, and reuse the serving window to classify article feeds.

## Conclusion

Deployment closes the CRISP-DM cycle for this project. The winning skip-gram classifier is packed with the exact training-time cleaning pipeline, verified by parity on the held-out set, and published as a public Gradio app on a Hugging Face ZeroGPU Space. The app accepts a news URL or pasted text and answers sport or finance with class scores, using the polite-request rules defined at the start of the project.